# 🎨 Visualization: t-SNE Embeddings on Roman-empire

**Milestone 3 — Analysis 1: Visualization**

Projects penultimate node embeddings from `GATv2`, `GraphSAGE`, `H2GCN`, and `MLP` into 2D using t-SNE on **Roman-empire** (largest dataset: 22,662 nodes, 18 classes, heterophilic).

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import inspect
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from sklearn.manifold import TSNE

from src.datasets import load_dataset
from src.models import create_model
from src.evaluate import extract_embeddings, compute_all_metrics
from src.utils import set_seed, get_device

device = get_device()
set_seed(42)

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.dpi'] = 150
os.makedirs('../results/plots', exist_ok=True)

VIZ_MODELS = ['gatv2', 'graphsage', 'h2gcn', 'mlp']
MODEL_DISPLAY = {'gatv2': 'GATv2', 'graphsage': 'GraphSAGE', 'h2gcn': 'H2GCN', 'mlp': 'MLP'}
MODEL_COLORS = {'gatv2': '#1b4965', 'graphsage': '#2a9d8f', 'h2gcn': '#e76f51', 'mlp': '#ca6702'}

## 1. Load Roman-empire & Trained Checkpoints

In [ ]:
data = load_dataset('roman-empire', root='../data')
print(f"Dataset: {data.dataset_name} ({data.num_nodes} nodes, {data.num_classes} classes)")

models = {}
for mname in VIZ_MODELS:
    ckpt_path = f'../results/models/roman-empire/{mname}/seed_42.pt'
    checkpoint = torch.load(ckpt_path, map_location=device, weights_only=False)
    params = dict(checkpoint.get('model_kwargs', {}))
    model = create_model(mname, data.num_node_features, data.num_classes, **params).to(device)
    model.load_state_dict(checkpoint['model_state'])
    model.eval()
    models[mname] = model
    print(f"Loaded {MODEL_DISPLAY[mname]} (val_acc: {checkpoint.get('best_val_accuracy', 0.0):.4f})")

## 2. Extract Embeddings & Run t-SNE

In [ ]:
data_dev = data.to(device)
sample_size = min(4000, data.num_nodes)
sample = np.random.default_rng(42).choice(data.num_nodes, sample_size, replace=False)

def make_tsne(perplexity=30, max_iter=1000, random_state=42):
    sig = inspect.signature(TSNE.__init__).parameters
    kw = dict(n_components=2, perplexity=perplexity, random_state=random_state, init='pca')
    kw['max_iter' if 'max_iter' in sig else 'n_iter'] = max_iter
    return TSNE(**kw)

tsne_results = {}
metrics_results = {}
for mname, model in models.items():
    embeddings = extract_embeddings(model, data_dev)
    tsne_results[mname] = make_tsne().fit_transform(embeddings[sample])
    metrics_results[mname] = compute_all_metrics(model, data_dev)
    m = metrics_results[mname]
    print(f"{MODEL_DISPLAY[mname]}: Test Acc={m['accuracy']:.4f}, Macro F1={m['macro_f1']:.4f}")

labels_sample = data.y.cpu().numpy()[sample]
test_mask_full = data.test_mask.cpu().numpy()
test_mask_sample = test_mask_full[sample]

## 3. t-SNE Plots (Ground-Truth Colored)

In [ ]:
fig, axes = plt.subplots(1, len(VIZ_MODELS), figsize=(5 * len(VIZ_MODELS), 5), constrained_layout=True)
for ax, mname in zip(axes, VIZ_MODELS):
    emb = tsne_results[mname]
    m = metrics_results[mname]
    ax.scatter(emb[:, 0], emb[:, 1], c=labels_sample, cmap='tab20', s=6, alpha=0.7)
    ax.set_title(f"{MODEL_DISPLAY[mname]}\nAcc: {m['accuracy']:.3f} | F1: {m['macro_f1']:.3f}", fontweight='bold')
    ax.set_xticks([]); ax.set_yticks([])

fig.suptitle('Roman-empire Learned Representations (t-SNE, 4,000 sampled nodes)', fontweight='bold')
fig.savefig('../results/plots/roman_embedding_tsne.png', dpi=180)
plt.show()

## 4. t-SNE with Misclassification Markers

In [ ]:
fig, axes = plt.subplots(1, len(VIZ_MODELS), figsize=(5 * len(VIZ_MODELS), 5), constrained_layout=True)
test_indices = np.where(test_mask_full)[0]

for ax, mname in zip(axes, VIZ_MODELS):
    emb = tsne_results[mname]
    m = metrics_results[mname]
    preds_full = np.full(data.num_nodes, -1)
    preds_full[test_indices] = m['predictions']
    preds_s = preds_full[sample]
    correct_s = (preds_s == labels_sample) & test_mask_sample
    wrong_s = (preds_s != labels_sample) & test_mask_sample
    non_test_s = ~test_mask_sample

    ax.scatter(emb[non_test_s, 0], emb[non_test_s, 1], c='lightgray', s=3, alpha=0.25)
    ax.scatter(emb[correct_s, 0], emb[correct_s, 1], c=labels_sample[correct_s], cmap='tab20', s=8, alpha=0.65)
    ax.scatter(emb[wrong_s, 0], emb[wrong_s, 1], c='red', s=18, marker='x', linewidths=0.8, label=f'Errors ({wrong_s.sum()})')
    ax.set_title(f"{MODEL_DISPLAY[mname]} (Acc: {m['accuracy']:.3f})", fontweight='bold')
    ax.legend(fontsize=8, loc='upper right')
    ax.set_xticks([]); ax.set_yticks([])

fig.suptitle('Roman-empire t-SNE with Misclassified Test Nodes (Red ×)', fontweight='bold')
fig.savefig('../results/plots/roman_tsne_misclassified.png', dpi=180)
plt.show()

## 5. Confusion Matrices & Per-Class F1

In [ ]:
fig, axes = plt.subplots(1, len(VIZ_MODELS), figsize=(5.2 * len(VIZ_MODELS), 4.8), constrained_layout=True)
for ax, mname in zip(axes, VIZ_MODELS):
    cm = metrics_results[mname]['confusion_matrix'].astype(float)
    cm_norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1.0)
    sns.heatmap(cm_norm, ax=ax, cmap='YlOrRd', vmin=0, vmax=1, cbar=(ax == axes[-1]))
    ax.set_title(MODEL_DISPLAY[mname], fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')

fig.suptitle('Normalized Confusion Matrices — Roman-empire (18 classes)', fontweight='bold')
fig.savefig('../results/plots/roman_confusion_matrices.png', dpi=180)
plt.show()

fig, ax = plt.subplots(figsize=(14, 5), constrained_layout=True)
x = np.arange(data.num_classes)
width = 0.8 / len(VIZ_MODELS)
for i, mname in enumerate(VIZ_MODELS):
    ax.bar(x + i * width, metrics_results[mname]['per_class_f1'], width, label=MODEL_DISPLAY[mname], color=MODEL_COLORS[mname], edgecolor='white')
ax.set_xticks(x + width * (len(VIZ_MODELS) - 1) / 2)
ax.set_xticklabels([f'C{c}' for c in range(data.num_classes)])
ax.set_ylabel('F1-Score'); ax.set_ylim(0, 1.05)
ax.set_title('Per-Class F1 Scores — Roman-empire', fontweight='bold')
ax.legend()
fig.savefig('../results/plots/roman_per_class_f1.png', dpi=180)
plt.show()